## Imports and Configs

In [2]:
import sys
from pathlib import Path
 
REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))
 
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
 
DATA_DIR    = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" #We'll store all the generated figs in this folder
FIGURES_DIR.mkdir(exist_ok=True)
 
device = "cuda" #No other option

#For reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

## Loading in the IOI dataset we built

In [12]:
data = torch.load(DATA_DIR / "ioi_dataset.pt", weights_only=False)
clean_toks = data["clean_toks"]            
corrupted_toks = data["corrupted_toks"]
clean_io_ids = data["clean_io_token_ids"]
clean_s_ids = data["clean_s_token_ids"]
corrupted_io_ids = data["corrupted_io_token_ids"]
corrupted_s_ids = data["corrupted_s_token_ids"]
end_positions = data["end_positions"]
num_prompts = data["N"]

## Loading in the Gemma model

In [4]:
model = HookedTransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.bfloat16,
)

model.eval() #As mentioned before, to handle the dropout layers
n_layers = model.cfg.n_layers
seq_len  = clean_toks.shape[1]
print(f"n_layers={n_layers}, d_model={model.cfg.d_model}, seq_len={seq_len}")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
n_layers=26, d_model=2304, seq_len=22


## Attribution Patching

In [5]:
def ioi_logit_diff(logits, io_ids, s_ids, ends):
    batch_idx    = torch.arange(logits.shape[0], device=logits.device)
    final_logits = logits[batch_idx, ends]
    io_l = final_logits.gather(1, io_ids.unsqueeze(1)).squeeze(1)
    s_l  = final_logits.gather(1, s_ids.unsqueeze(1)).squeeze(1)
    return (io_l - s_l).sum()

In [10]:
HOOK_NAMES = [f"blocks.{L}.hook_resid_post" for L in range(n_layers)]
BATCH_SIZE = 1
attribution_sum = torch.zeros(n_layers, seq_len, device=device, dtype=torch.float32)
clean_metric_sum = 0.0
corrupted_metric_sum = 0.0

In [14]:
for i in range(0, 5, BATCH_SIZE):
    j = min(i + BATCH_SIZE, N)
    bsz = j - i
 
    # Move just this batch to GPU
    c_toks   = clean_toks[i:j].to(device)
    cor_toks = corrupted_toks[i:j].to(device)
    c_io     = clean_io_ids[i:j].to(device)
    c_s      = clean_s_ids[i:j].to(device)
    cor_io   = corrupted_io_ids[i:j].to(device)
    cor_s    = corrupted_s_ids[i:j].to(device)
    ends     = end_positions[i:j].to(device)
 
    # ---- Pass 1: clean forward, cache resid_post (no grad) ----
    clean_cache = {}
    def clean_hook(act, hook):
        clean_cache[hook.name] = act.detach()
        return act
    with torch.no_grad():
        clean_logits = model.run_with_hooks(
            c_toks, fwd_hooks=[(name, clean_hook) for name in HOOK_NAMES]
        )
    clean_metric_sum += ioi_logit_diff(clean_logits, c_io, c_s, ends).item()
    del clean_logits
 
    # ---- Pass 2: corrupted forward, cache resid_post (no grad) ----
    corrupted_cache = {}
    def corr_hook(act, hook):
        corrupted_cache[hook.name] = act.detach()
        return act
    with torch.no_grad():
        cor_logits = model.run_with_hooks(
            cor_toks, fwd_hooks=[(name, corr_hook) for name in HOOK_NAMES]
        )
    corrupted_metric_sum += ioi_logit_diff(cor_logits, cor_io, cor_s, ends).item()
    del cor_logits
 
    # ---- Pass 3: corrupted forward + backward, capture gradients ----
    grad_cache = {}
    def grad_hook(act, hook):
        act.requires_grad_(True)
        act.retain_grad()
        grad_cache[hook.name] = act
        return act
    cor_logits_grad = model.run_with_hooks(
        cor_toks, fwd_hooks=[(name, grad_hook) for name in HOOK_NAMES]
    )
    metric = ioi_logit_diff(cor_logits_grad, cor_io, cor_s, ends)
    metric.backward()
    del cor_logits_grad, metric
 
    # ---- Combine: (clean - corrupted) * grad, sum over d_model, sum over batch ----
    # We sum (not mean) over the batch dim and divide by N at the end. This way
    # accumulating across batches is just pointwise addition.
    for L, name in enumerate(HOOK_NAMES):
        diff = (clean_cache[name] - corrupted_cache[name]).float()    # [B, seq, d_model]
        grad = grad_cache[name].grad.detach().float()                  # [B, seq, d_model]
        attribution_sum[L] += (diff * grad).sum(dim=-1).sum(dim=0)    # [seq]
 
    # Free everything before the next batch
    del clean_cache, corrupted_cache, grad_cache
    del c_toks, cor_toks, c_io, c_s, cor_io, cor_s, ends
    torch.cuda.empty_cache()
 
    if (i // BATCH_SIZE) % 4 == 0:
        print(f"  batch {i//BATCH_SIZE + 1}/{(N + BATCH_SIZE - 1)//BATCH_SIZE} "
              f"(prompts {i}-{j}) — VRAM allocated: "
              f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

  batch 1/5000 (prompts 0-1) — VRAM allocated: 14.81 GB


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.10 GiB. GPU 0 has a total capacity of 15.48 GiB of which 1.08 GiB is free. Including non-PyTorch memory, this process has 14.39 GiB memory in use. Of the allocated memory 13.79 GiB is allocated by PyTorch, and 449.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
 

 
# Finalize: divide by N to convert "sum over prompts" → "mean per prompt"
attribution = (attribution_sum / N).cpu()
 
# Sanity check the metric values against Day 1
print(f"\nVerify clean metric:     {clean_metric_sum / N:+.3f} "
      f"(Day 1 baseline: {pilot['baseline_logit_diff']:+.3f})")
print(f"Verify corrupted metric: {corrupted_metric_sum / N:+.3f}")
 
print(f"\nAttribution shape: {tuple(attribution.shape)}")
print(f"Range: [{attribution.min().item():.3f}, {attribution.max().item():.3f}]")
 
# Save raw attribution
torch.save({
    "attribution":     attribution,
    "hook_names":      HOOK_NAMES,
    "clean_logit_diff":     pilot["baseline_logit_diff"],
    "n_prompts":       N,
    "batch_size":      BATCH_SIZE,
}, DATA_DIR / "ap_residual_attribution.pt")
print(f"Saved raw attribution to {DATA_DIR / 'ap_residual_attribution.pt'}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
vmax = attribution.abs().max().item()
im = ax.imshow(
    attribution.numpy(),
    aspect="auto", cmap="RdBu_r",
    vmin=-vmax, vmax=vmax,
    interpolation="nearest",
)
ax.set_xlabel("Token position")
ax.set_ylabel("Layer")
ax.set_title("Residual-stream attribution patching (IOI on Gemma-2-2B)")
ax.set_yticks(range(n_layers))
ax.set_xticks(range(seq_len))
plt.colorbar(im, ax=ax, label="Attribution (mean per prompt)")
plt.tight_layout()
fig_path = FIGURES_DIR / "ap_residual_heatmap.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"\nSaved heatmap to {fig_path}")
plt.show()


In [ ]:
print("\nTop 10 (layer, position) pairs by |attribution|:")
flat = attribution.abs().flatten()
top_k = torch.topk(flat, 10)
for rank, (val, idx) in enumerate(zip(top_k.values, top_k.indices), 1):
    L, P = divmod(idx.item(), seq_len)
    signed = attribution[L, P].item()
    print(f"  {rank:2d}. layer={L:2d}, pos={P:2d}   attribution={signed:+.3f}")
 
print("\nDay 2 (AP baseline) complete.")
